In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

c:\Users\manh\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
df = pd.read_csv("./vn30_filtered.csv")
df

,symbol,time,close,volume,vnindex
0,ACB,2017-01-03,3.33,1609757,672.01
1,ACB,2017-01-04,3.35,739000,674.70
2,ACB,2017-01-05,3.35,471881,675.81
3,ACB,2017-01-06,3.53,1409824,679.80
4,ACB,2017-01-09,3.65,970791,682.57
...,...,...,...,...,...
53045,VRE,2025-12-25,32.25,17089700,1742.85
53046,VRE,2025-12-26,32.00,20927400,1729.80
53047,VRE,2025-12-29,33.20,7573400,1754.84
53048,VRE,2025-12-30,32.80,8839700,1766.90


# Add feature

In [16]:
def compute_log_return(df=None, col_name=None, computed_col=None, group_col="symbol", time_col="time"):
    df = df.copy()

    df[time_col] = pd.to_datetime(df[time_col])
    df = df.sort_values(by=[group_col, time_col])

    df[computed_col] = df[computed_col].replace(0, np.nan)
    
    df[col_name] = df.groupby(group_col)[computed_col].transform(
        lambda x: np.log(x / x.shift(1))
    )

    return df

In [17]:
df2 = compute_log_return(df, col_name="log_ret", computed_col="close")
df2 = compute_log_return(df2, col_name="vni_log_ret", computed_col="vnindex")
df2

,symbol,time,close,volume,vnindex,log_ret,vni_log_ret
0,ACB,2017-01-03,3.33,1609757,672.01,NaN,NaN
1,ACB,2017-01-04,3.35,739000,674.70,0.005988,0.003995
2,ACB,2017-01-05,3.35,471881,675.81,0.000000,0.001644
3,ACB,2017-01-06,3.53,1409824,679.80,0.052338,0.005887
4,ACB,2017-01-09,3.65,970791,682.57,0.033429,0.004066
...,...,...,...,...,...,...,...
53045,VRE,2025-12-25,32.25,17089700,1742.85,-0.071780,-0.022675
53046,VRE,2025-12-26,32.00,20927400,1729.80,-0.007782,-0.007516
53047,VRE,2025-12-29,33.20,7573400,1754.84,0.036814,0.014372
53048,VRE,2025-12-30,32.80,8839700,1766.90,-0.012121,0.006849


In [18]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53050 entries, 0 to 53049
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   symbol       53050 non-null  object        
 1   time         53050 non-null  datetime64[ns]
 2   close        53050 non-null  float64       
 3   volume       53050 non-null  int64         
 4   vnindex      53050 non-null  float64       
 5   log_ret      53026 non-null  float64       
 6   vni_log_ret  53026 non-null  float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(1)
memory usage: 2.8+ MB


# Add label

In [5]:
def add_target(df, horizon=1):
    df = df.copy()
    df["target"] = df.groupby("symbol")["log_return"].shift(-horizon)
    df['target_time'] = df.groupby('symbol')['time'].shift(-horizon)
    df = df.dropna(subset=["target"])
    return df

In [6]:
df3 = add_target(df2)
df3

,symbol,time,close,log_return,target,target_time
1,ACB,2017-01-04,3.35,0.005988,0.000000,2017-01-05
2,ACB,2017-01-05,3.35,0.000000,0.052338,2017-01-06
3,ACB,2017-01-06,3.53,0.052338,0.033429,2017-01-09
4,ACB,2017-01-09,3.65,0.033429,0.010899,2017-01-10
5,ACB,2017-01-10,3.69,0.010899,0.026740,2017-01-11
...,...,...,...,...,...,...
53067,VRE,2025-12-24,34.65,0.011611,-0.071780,2025-12-25
53068,VRE,2025-12-25,32.25,-0.071780,-0.007782,2025-12-26
53069,VRE,2025-12-26,32.00,-0.007782,0.036814,2025-12-29
53070,VRE,2025-12-29,33.20,0.036814,-0.012121,2025-12-30


In [7]:
df3.symbol.nunique()

24

In [8]:
df3.describe()

,time,close,log_return,target,target_time
count,53025,53025.000000,53025.000000,53025.000000,53025
mean,2021-07-28 12:51:20.418670336,37.425171,0.000655,0.000656,2021-07-30 00:04:28.854313984
min,2017-01-04 00:00:00,1.390000,-0.164622,-0.164622,2017-01-05 00:00:00
25%,2019-05-20 00:00:00,12.570000,-0.008917,-0.008920,2019-05-21 00:00:00
50%,2021-07-30 00:00:00,27.500000,0.000000,0.000000,2021-08-02 00:00:00
75%,2023-10-16 00:00:00,57.310000,0.009901,0.009901,2023-10-17 00:00:00
max,2025-12-30 00:00:00,219.100000,1.304007,1.304007,2025-12-31 00:00:00
std,NaN,31.134741,0.021758,0.021756,NaN


# Train/Val/Test

In [9]:
def split_time_series(df,
                      train_start='2018-01-01',
                      train_end='2023-01-01',
                      val_end='2024-01-01',
                      test_end=None,
                      time_col='time'):
    
    df = df.copy()

    train_df = df[
        (df[time_col] >= train_start) &
        (df[time_col] < train_end)
    ]

    val_df = df[
        (df[time_col] >= train_end) &
        (df[time_col] < val_end)
    ]

    if test_end is not None:
        test_df = df[
            (df[time_col] >= val_end) &
            (df[time_col] < test_end)
        ]
    else:
        test_df = df[df[time_col] >= val_end]

    return train_df, val_df, test_df

In [10]:
train_df, val_df, test_df = split_time_series(df3)
train_df.shape, val_df.shape, test_df.shape

((29904, 6), (5976, 6), (11952, 6))

In [11]:
print(train_df['time'].min(), train_df['time'].max())
print(val_df['time'].min(), val_df['time'].max())
print(test_df['time'].min(), test_df['time'].max())

2018-01-02 00:00:00 2022-12-30 00:00:00
2023-01-03 00:00:00 2023-12-29 00:00:00
2024-01-02 00:00:00 2025-12-30 00:00:00


In [12]:
print(train_df["symbol"].nunique())
print(val_df["symbol"].nunique())
print(test_df["symbol"].nunique())

24
24
24


In [13]:
set(test_df.symbol.unique()) - set(train_df.symbol.unique())

set()

In [14]:
symbols_in_train = train_df['symbol'].unique()

test_df = test_df[test_df['symbol'].isin(symbols_in_train)]

test_df['symbol'].nunique(), test_df.shape

(24, (11952, 6))

# Create sequences

In [15]:
def create_sequences(df, window=20, feature_col='log_return', target_col='target'):
    X, y, meta = [], [], [] 
    
    for symbol, sub_df in df.groupby('symbol'):
        sub_df = sub_df.sort_values('time').reset_index(drop=True)

        values_x = sub_df[feature_col].values
        values_y = sub_df[target_col].values
        
        for i in range(len(sub_df) - window):
            X.append(values_x[i : i + window])
            y.append(values_y[i + window - 1])
            meta.append({
                'symbol': symbol,
                'time': sub_df['target_time'].iloc[i + window - 1]
            })
    
    X = np.array(X).reshape(-1, window, 1)
    y = np.array(y)
    meta_df = pd.DataFrame(meta)
    
    return X, y, meta_df

In [16]:
X_train, y_train, meta_train = create_sequences(train_df)
X_train.shape, y_train.shape, meta_train.shape

((29424, 20, 1), (29424,), (29424, 2))

In [17]:
X_val, y_val, meta_val = create_sequences(val_df)
X_val.shape, y_val.shape, meta_val.shape

((5496, 20, 1), (5496,), (5496, 2))

In [18]:
X_test, y_test, meta_test = create_sequences(test_df)
X_test.shape, y_test.shape, meta_test.shape

((11472, 20, 1), (11472,), (11472, 2))

In [19]:
34511+20*29, 6641+20*29, 13862+20*29

(35091, 7221, 14442)

# Building LSTM